# KITScenes Phase 6 — Stage A OneAlign Parity & Stage B Candidates
Runs Stage A (`8-bit` vs lossless `fp16` on T4/L4) and Stage B (`cand_a_baseline`..`cand_e_selective` with inference-time Ridge trajectories) cleanly on Google Colab.

In [ ]:
#@title 1. Setup Environment, Checkout `phase6-onealign-candidates`, & Fetch `videos_submission.bin`
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
import torch

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

subprocess.run(["nvidia-smi"], check=False)
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
tot_vram = sum(torch.cuda.get_device_properties(i).total_memory / (1024**3) for i in range(n_gpus))
print(f"PyTorch={torch.__version__} | GPUs={n_gpus} | total_VRAM={tot_vram:.2f} GiB", flush=True)

repo_dir = Path("/content/kitscenes-world-model")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "phase6-onealign-candidates"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "phase6-onealign-candidates"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/phase6-onealign-candidates"], check=True)
else:
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "--branch", "phase6-onealign-candidates",
            "https://github.com/wageshsharma20/kitscenes-world-model.git",
            str(repo_dir),
        ],
        check=True,
    )
commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()
print(f"Checked out phase6-onealign-candidates @ {commit}", flush=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv", "kaggle"], check=True)
venv_dir = Path("/tmp/qalign_venv")
if not (venv_dir / "bin" / "python").exists():
    subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)], check=True)
q_pip = str(venv_dir / "bin" / "pip")
subprocess.run(
    [
        q_pip, "install", "-q", "--no-deps", "--force-reinstall",
        "transformers==4.36.1",
        "tokenizers==0.15.2",
        "huggingface-hub==0.25.2",
        "accelerate==0.29.3",
        "bitsandbytes>=0.50.2",
    ],
    check=True,
)
subprocess.run([q_pip, "install", "-q", "icecream", "sentencepiece", "protobuf"], check=True)

# Configure Kaggle auth from Colab Secrets or upload kaggle.json only if archive isn't cached yet
def _find_archive():
    for cand in [
        Path("/content/videos.zip"),
        Path("/content/videos_submission.bin"),
        Path("/content/kaggle_dl/videos_submission.bin"),
    ]:
        if cand.exists() and cand.stat().st_size > 2_000_000_000:
            return cand
    return None

zip_path = _find_archive()
if zip_path is None:
    try:
        from google.colab import userdata
        k_tok = os.environ.get("KAGGLE_API_TOKEN") or ""
        try:
            k_tok = k_tok or userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            pass
        if k_tok:
            os.environ["KAGGLE_API_TOKEN"] = k_tok
        else:
            try:
                u, k = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
                if u and k:
                    kdir = Path("/root/.kaggle")
                    kdir.mkdir(exist_ok=True)
                    kfile = kdir / "kaggle.json"
                    kfile.write_text(json.dumps({"username": u, "key": k}))
                    kfile.chmod(0o600)
            except Exception:
                pass
    except Exception:
        pass

    kaggle_cfg = Path("/root/.kaggle/kaggle.json")
    if not kaggle_cfg.exists() and not os.environ.get("KAGGLE_API_TOKEN"):
        from google.colab import files
        print("Upload kaggle.json once to download wageshsh/kitscenes-prod-videos-final cleanly:")
        uploaded = files.upload()
        for fname, raw in uploaded.items():
            if fname.endswith(".json"):
                kaggle_cfg.parent.mkdir(exist_ok=True)
                kaggle_cfg.write_bytes(raw)
                kaggle_cfg.chmod(0o600)

    dl_dir = Path("/content/kaggle_dl")
    dl_dir.mkdir(exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "wageshsh/kitscenes-prod-videos-final", "-p", str(dl_dir), "--unzip"],
        check=True,
    )
    zip_path = _find_archive()

assert zip_path is not None and zip_path.exists(), "Submission archive not found!"
print(f"Ready: {zip_path} ({zip_path.stat().st_size / (1024**3):.2f} GiB)", flush=True)


In [ ]:
#@title 2. Run Stage A Parity Test (8-bit vs FP16) — Start with 12, then set N_CLIPS=48
N_CLIPS = 12  #@param [12, 48] {type:"raw"}

import json, os, subprocess
from pathlib import Path

repo_dir = Path("/content/kitscenes-world-model")
q_py = "/tmp/qalign_venv/bin/python"
zip_path = next(p for p in [Path("/content/videos.zip"), Path("/content/videos_submission.bin"), Path("/content/kaggle_dl/videos_submission.bin")] if p.exists())

out_8bit = Path(f"/content/stage_a_{N_CLIPS}_t4_8bit.json")
out_fp16 = Path(f"/content/stage_a_{N_CLIPS}_fp16.json")
out_cmp = Path(f"/content/stage_a_{N_CLIPS}_parity.json")

subprocess.run(
    [
        q_py, str(repo_dir / "scripts" / "smoke_test_onealign.py"), "score",
        "--videos-zip", str(zip_path),
        "--limit", str(N_CLIPS),
        "--precision", "8bit",
        "--out", str(out_8bit),
    ],
    check=True,
)
subprocess.run(
    [
        q_py, str(repo_dir / "scripts" / "smoke_test_onealign.py"), "score",
        "--videos-zip", str(zip_path),
        "--limit", str(N_CLIPS),
        "--precision", "fp16",
        "--out", str(out_fp16),
    ],
    check=True,
)
cmp_proc = subprocess.run(
    [
        q_py, str(repo_dir / "scripts" / "smoke_test_onealign.py"), "compare",
        "--eightbit", str(out_8bit),
        "--fp16", str(out_fp16),
        "--metadata", str(repo_dir / "data" / "history.jsonl"),
        "--out", str(out_cmp),
    ],
    check=False,
)
print("PARITY COMPARE EXIT:", cmp_proc.returncode, flush=True)
summary = json.loads(out_cmp.read_text())
print(json.dumps({k: v for k, v in summary.items() if k != "per_clip"}, indent=2))


In [ ]:
#@title 3. Stage B Candidate Comparison (Run AFTER Stage A Parity Passes)
LIMIT_SCENARIOS = 30  #@param [30, 100] {type:"raw"}

import json, os, subprocess, sys
from pathlib import Path

repo_dir = Path("/content/kitscenes-world-model")
q_py = "/tmp/qalign_venv/bin/python"
val_out = Path("/content/val_extract")
val_out.mkdir(exist_ok=True)
if not list(val_out.rglob("val_calib.tar")):
    subprocess.run(["kaggle", "kernels", "output", "wageshsh/kitscenes-extract-cpu", "-p", str(val_out)], check=True)
val_tar = next(val_out.rglob("val_calib.tar"))

calib_dir = Path("/content/stage_b_calib")
subprocess.run(["bash", "-lc", "which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "simple-lama-inpainting"], check=True)

subprocess.run(
    [
        sys.executable, str(repo_dir / "scripts" / "eval_world_model_qalign.py"),
        "--stage", "render",
        "--val-tar", str(val_tar),
        "--out-dir", str(calib_dir),
        "--variants", "cand_a_baseline,cand_b_scale1,cand_c_adaptive,cand_d_photometric,cand_e_selective",
        "--traj-source", "ridge",
        "--crf", "14",
        "--limit", str(LIMIT_SCENARIOS),
        "--device", "cuda:0",
    ],
    check=True,
)
subprocess.run(
    [
        q_py, str(repo_dir / "scripts" / "eval_world_model_qalign.py"),
        "--stage", "score",
        "--out-dir", str(calib_dir),
        "--baseline-variant", "cand_a_baseline",
    ],
    check=True,
)
print((calib_dir / "calibration_summary.json").read_text()[:4000])
